In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
df.shape

(2000, 8)

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [6]:
df=df.drop('id',axis=1)

In [7]:
answer=df['answer']
df=df.drop('answer',axis=1)

In [8]:
df.shape

(2000, 6)

In [9]:
"""import re

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r'[^a-zA-Z0-9\s]',' ',text)
    text=re.sub(r'\s+',' ',text).strip()
    return text

df['prompt']=df['prompt'].apply(clean_text)
df['A']=df['A'].apply(clean_text)
df['B']=df['B'].apply(clean_text)
df['C']=df['C'].apply(clean_text)
df['D']=df['D'].apply(clean_text)
df['E']=df['E'].apply(clean_text)"""

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_23/643210556.py:5: SyntaxWarning: invalid escape sequence '\s'
  text=re.sub(r'[^a-zA-Z0-9\s]',' ',text)


"import re\n\ndef clean_text(text):\n    text=str(text).lower()\n    text=re.sub(r'[^a-zA-Z0-9\\s]',' ',text)\n    text=re.sub(r'\\s+',' ',text).strip()\n    return text\n\ndf['prompt']=df['prompt'].apply(clean_text)\ndf['A']=df['A'].apply(clean_text)\ndf['B']=df['B'].apply(clean_text)\ndf['C']=df['C'].apply(clean_text)\ndf['D']=df['D'].apply(clean_text)\ndf['E']=df['E'].apply(clean_text)"

In [10]:
df['prompt_len']=df['prompt'].str.len()
for col in ['A','B','C','D','E']:
    df[f'{col}_len']=df[col].str.len()

def get_overlaps(row):
    prompt_words=set(row['prompt'].split())
    overlaps={}
    for col in ['A','B','C','D','E']:
        option_words=set(row[col].split())
        overlaps[f'{col}_overlap']=len(prompt_words & option_words)
    return pd.Series(overlaps)

overlaps_df=df.apply(get_overlaps,axis=1)
df=pd.concat([df,overlaps_df],axis=1)

In [11]:
from sklearn.model_selection import train_test_split

x_train,x_val,y_train,y_val=train_test_split(df,answer,train_size=0.8,random_state=1)

In [12]:
def score_to_labels(scores,top1=False):
    labels=[]
    top1_labels=[]
    for score in scores:
        scored=np.argsort(score)[::-1][:3]
        top_3=[chr(65+i) for i in scored]
        labels.append(" ".join(top_3))
        top1_labels.append(chr(65+np.argsort(score)[::-1][0]))
    if not top1:
        return labels
    else:
        return top1_labels

In [13]:
def map_at_3(predictions,ground_truth):
    ap_scores=[]
    for pred, truth in zip(predictions, ground_truth):
        pred_list = pred.split()
        if truth in pred_list:
            ap_scores.append(1.0 / (pred_list.index(truth) + 1))
        else:
            ap_scores.append(0.0)
    return sum(ap_scores) / len(ap_scores)

In [14]:
from sklearn.metrics import accuracy_score,f1_score

def support_metrics(train_scores,val_scores):
    train_acc=accuracy_score(score_to_labels(train_scores,top1=True),y_train)
    val_acc=accuracy_score(score_to_labels(val_scores,top1=True),y_val)
    train_f1=f1_score(score_to_labels(train_scores,top1=True),y_train,average='macro')
    val_f1=f1_score(score_to_labels(val_scores,top1=True),y_val,average='macro')
    return train_acc,val_acc,train_f1,val_f1

In [15]:
!pip install wandb -q

In [16]:
import wandb
from kaggle_secrets import UserSecretsClient
import os

secrets=UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login()
wandb_mode=True

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [17]:
def len_scores(row):
    lengths=[row['A_len'],row['B_len'],row['C_len'],row['D_len'],row['E_len']]
    max_len=max(lengths)
    min_len=min(lengths)
    scores=[]
    for length in lengths:
        score=0
        if length==max_len:
            score=0.5
        elif length==min_len:
            score=0.2
        else:
            score=0.3
        scores.append(score)
    return scores

length_train_scores=x_train.apply(len_scores,axis=1).to_list()
length_labels=score_to_labels(length_train_scores)
length_map3=map_at_3(length_labels,y_train)
print(length_map3)

0.5320833333333334


In [18]:
length_val_scores=x_val.apply(len_scores,axis=1).to_list()
length_val_labels=score_to_labels(length_val_scores)
length_val_map3=map_at_3(length_val_labels,y_val)
length_val_map3

0.49666666666666665

In [19]:
from sklearn.metrics import accuracy_score,f1_score

len_train_acc,len_val_acc,len_train_f1,len_val_f1=support_metrics(length_train_scores,length_val_scores)

In [20]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="largestlen-baseline-metrics",id="largestlen_v1",resume="allow")
    wandb.log({
        "map_at_3_train":length_map3,   
        "map_at_3_val":length_val_map3,
        "accuracy_at_train":len_train_acc,
        "accuracy_at_val":len_val_acc,
        "f1_at_train":len_train_f1,
        "f1_at_val":len_val_f1,
        "metric":"Longest-Length"
        })
    wandb.finish()

wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260706_190339-largestlen_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run largestlen-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/largestlen_v1
wandb: updating run metadata
wandb: uploading history steps 8-8, summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.405
wandb:   accuracy_at_val 0.3825
wandb:       f1_at_train 0.40268
wandb:         f1_at_val 0.37871
wandb:    map_at_3_train 0.53208
wandb:      map_at_3_val 0.49667
wandb:            metric Longest-Length
wandb:

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True)
all_text=df['prompt'].tolist()+df['A'].tolist()+df['B'].tolist()+df['C'].tolist()+df['D'].tolist()+df['E'].tolist()
tfidf.fit(all_text)

TfidfVectorizer(sublinear_tf=True)

In [22]:
tfidf_train_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_train['prompt']+" "+x_train[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_train_scores.append(np.asarray(scores).flatten())
tfidf_train_scores=np.array(tfidf_train_scores).T

In [23]:
tfidf_val_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_val['prompt']+" "+x_val[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_val_scores.append(np.asarray(scores).flatten())
tfidf_val_scores=np.array(tfidf_val_scores).T

In [24]:
tfidf_train_labels=score_to_labels(tfidf_train_scores,top1=False)
tfidf_train_map3=map_at_3(tfidf_train_labels,y_train)
print(tfidf_train_map3)

tfidf_val_labels=score_to_labels(tfidf_val_scores,top1=False)
tfidf_val_map3=map_at_3(tfidf_val_labels,y_val)
print(tfidf_val_map3)

0.524375
0.5133333333333333


In [25]:
tfidf_train_acc,tfidf_val_acc,tfidf_train_f1,tfidf_val_f1=support_metrics(tfidf_train_scores,tfidf_val_scores)

In [26]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="tfidf-baseline-metrics",id="tfidf_v1",resume="allow")
    wandb.log({
        "map_at_3_train":tfidf_train_map3,   
        "map_at_3_val":tfidf_val_map3,
        "accuracy_at_train":tfidf_train_acc,
        "accuracy_at_val":tfidf_val_acc,
        "f1_at_train":tfidf_train_f1,
        "f1_at_val":tfidf_val_f1,
        "vectorizer":"TfidfVectorizer",
        "sublinear_tf":True})
    wandb.finish()

wandb: setting up run tfidf_v1
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260706_190344-tfidf_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run tfidf-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/tfidf_v1
wandb: updating run metadata
wandb: uploading history steps 26-26, summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.33375
wandb:   accuracy_at_val 0.3375
wandb:       f1_at_train 0.33152
wandb:         f1_at_val 0.33492
wandb:    map_at_3_train 0.52438
wandb:      map_at_3_val 0.51333
wandb:      sublinear_tf T

In [27]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=10,window=1,min_count=1,seed=1,workers=1,epochs=10)

In [28]:
option_cols=['A','B','C','D','E']
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [29]:
prompts=np.vstack(x_train['prompt'].apply(sentence_embedding))
w2v_train_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_train[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_train_scores.append(scores)
w2v_train_scores=np.array(w2v_train_scores).T

In [30]:
val_prompts=np.vstack(x_val['prompt'].apply(sentence_embedding))
w2v_val_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_val[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(val_prompts,option_matrix))
    w2v_val_scores.append(scores)
w2v_val_scores=np.array(w2v_val_scores).T

In [31]:
w2v_train_labels=score_to_labels(w2v_train_scores)
w2v_train_map3=map_at_3(w2v_train_labels,y_train)
print(w2v_train_map3)

w2v_val_labels=score_to_labels(w2v_val_scores)
w2v_val_map3=map_at_3(w2v_val_labels,y_val)
print(w2v_val_map3)

0.37104166666666666
0.37083333333333335


In [32]:
w2v_train_acc,w2v_val_acc,w2v_train_f1,w2v_val_f1=support_metrics(w2v_train_scores,w2v_val_scores)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="word2vec-baseline-metrics",id="w2v_v1", resume="allow")
    wandb.log({
        "map_at_3_train":w2v_train_map3,   
        "map_at_3_val":w2v_val_map3,
        "accuracy_at_train":w2v_train_acc,
        "accuracy_at_val":w2v_val_acc,
        "f1_at_train":w2v_train_f1,
        "f1_at_val":w2v_val_f1,
        "embedding_model":"Word2Vec",
        "vector_size":20,
         "window":1,
         "min_count":1})
    wandb.finish()

wandb: setting up run w2v_v1
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260706_190414-w2v_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run word2vec-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/w2v_v1
wandb: updating run metadata
wandb: uploading requirements.txt; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb:         min_count ▁
wandb:       vector_size ▁
wandb:            window ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.22125
wandb:   accuracy_at_val 0.215
wandb:   embedding_model Word2Vec
wandb:       f1_at_train 0.21

In [33]:
from transformers import AutoModel,AutoTokenizer
import torch

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer1=AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
tokenizer2=AutoTokenizer.from_pretrained("roberta-base")
tokenizer3=AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

model1=AutoModel.from_pretrained("microsoft/deberta-v3-base")
model2=AutoModel.from_pretrained("roberta-base")
model3=AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

model1.to(device)
model2.to(device)
model3.to(device)
model1.eval()
model2.eval()
model2.eval()

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (15 outputs left out)

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [34]:
def get_embedding(text,tokenizer,model):
    if not text or not text.strip():
        return np.zeros(model.config.hidden_size)
    inputs=tokenizer(text,return_tensors='pt',max_length=128,truncation=True,padding=True)
    inputs={k:v.to(device) for k,v in inputs.items()}
    with torch.no_grad():
        outputs=model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

def get_scores(data,tokenizer,model):
    scores=[]
    prompts=np.vstack(data['prompt'].apply(lambda x:get_embedding(x,tokenizer,model)))
    for col in option_cols:
        options=np.vstack(data[col].apply(lambda x:get_embedding(x,tokenizer,model)))
        score=np.diag(cosine_similarity(prompts,options))
        scores.append(score)
    scores=np.array(scores).T
    labels=score_to_labels(scores)
    return scores,labels

In [35]:
"""bert_train_scores,bert_train_labels=get_scores(x_train,tokenizer1,model1)
bert_train=map_at_3(bert_train_labels,y_train)
bert_val_scores,bert_val_labels=get_scores(x_val,tokenizer1,model1)
bert_val=map_at_3(bert_val_labels,y_val)"""

'bert_train_scores,bert_train_labels=get_scores(x_train,tokenizer1,model1)\nbert_train=map_at_3(bert_train_labels,y_train)\nbert_val_scores,bert_val_labels=get_scores(x_val,tokenizer1,model1)\nbert_val=map_at_3(bert_val_labels,y_val)'

In [36]:
def get_embedding(text,tokenizer,model):
    if not text or not text.strip():
        return np.zeros(model.config.hidden_size)
    inputs=tokenizer(text,return_tensors='pt',max_length=128,truncation=True,padding=True)
    inputs={k:v.to(device) for k,v in inputs.items()}
    with torch.no_grad():
        outputs=model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

def get_scores(data,tokenizer,model):
    scores=[]
    prompts=np.vstack(data['prompt'].apply(lambda x:get_embedding(x,tokenizer,model)))
    for col in option_cols:
        options=np.vstack(data[col].apply(lambda x:get_embedding(x,tokenizer,model)))
        score=np.diag(cosine_similarity(prompts,options))
        scores.append(score)
    scores=np.array(scores).T
    labels=score_to_labels(scores)
    return scores,labels

"""roberta_train_scores,roberta_train_labels=get_scores(x_train,tokenizer2,model2)
roberta_train=map_at_3(roberta_train_labels,y_train)
roberta_val_scores,roberta_val_labels=get_scores(x_val,tokenizer2,model2)
roberta_val=map_at_3(roberta_val_labels,y_val)
st_train_scores,st_train_labels=get_scores(x_train,tokenizer3,model3)
st_train=map_at_3(st_train_labels,y_train)
st_val_scores,st_val_labels=get_scores(x_val,tokenizer3,model3)
st_val=map_at_3(st_val_labels,y_val)

print(bert_train)
print(bert_val)
print(roberta_train)
print(roberta_val)
print(st_train)
print(st_val) """

'roberta_train_scores,roberta_train_labels=get_scores(x_train,tokenizer2,model2)\nroberta_train=map_at_3(roberta_train_labels,y_train)\nroberta_val_scores,roberta_val_labels=get_scores(x_val,tokenizer2,model2)\nroberta_val=map_at_3(roberta_val_labels,y_val)\nst_train_scores,st_train_labels=get_scores(x_train,tokenizer3,model3)\nst_train=map_at_3(st_train_labels,y_train)\nst_val_scores,st_val_labels=get_scores(x_val,tokenizer3,model3)\nst_val=map_at_3(st_val_labels,y_val)\n\nprint(bert_train)\nprint(bert_val)\nprint(roberta_train)\nprint(roberta_val)\nprint(st_train)\nprint(st_val) '

In [37]:
"""st_train_scores,st_train_labels=get_scores(x_train,tokenizer3,model3)
st_train=map_at_3(st_train_labels,y_train)
st_val_scores,st_val_labels=get_scores(x_val,tokenizer3,model3)
st_val=map_at_3(st_val_labels,y_val)"""

'st_train_scores,st_train_labels=get_scores(x_train,tokenizer3,model3)\nst_train=map_at_3(st_train_labels,y_train)\nst_val_scores,st_val_labels=get_scores(x_val,tokenizer3,model3)\nst_val=map_at_3(st_val_labels,y_val)'

In [38]:
"""print(st_train)
print(st_val)"""

'print(st_train)\nprint(st_val)'

In [39]:
"""bert_train_acc,bert_val_acc,bert_train_f1,bert_val_f1=support_metrics(bert_train_scores,bert_val_scores)
roberta_train_acc,roberta_val_acc,roberta_train_f1,roberta_val_f1=support_metrics(roberta_train_scores,roberta_val_scores)
st_train_acc,st_val_acc,st_train_f1,st_val_f1=support_metrics(st_train_scores,st_val_scores)"""

'bert_train_acc,bert_val_acc,bert_train_f1,bert_val_f1=support_metrics(bert_train_scores,bert_val_scores)\nroberta_train_acc,roberta_val_acc,roberta_train_f1,roberta_val_f1=support_metrics(roberta_train_scores,roberta_val_scores)\nst_train_acc,st_val_acc,st_train_f1,st_val_f1=support_metrics(st_train_scores,st_val_scores)'

In [40]:
"""if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="BertEmbedding-metrics",id="bert_v1", resume="allow")
    wandb.log({
        "map_at_3_train":bert_train,   
        "map_at_3_val":bert_val,
        "accuracy_at_train":bert_train_acc,
        "accuracy_at_val":bert_val_acc,
        "f1_at_train":bert_train_f1,
        "f1_at_val":bert_val_f1,
        "embedding_model":"BERT",
        "max_length":128})
    wandb.finish()"""

'if wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="BertEmbedding-metrics",id="bert_v1", resume="allow")\n    wandb.log({\n        "map_at_3_train":bert_train,   \n        "map_at_3_val":bert_val,\n        "accuracy_at_train":bert_train_acc,\n        "accuracy_at_val":bert_val_acc,\n        "f1_at_train":bert_train_f1,\n        "f1_at_val":bert_val_f1,\n        "embedding_model":"BERT",\n        "max_length":128})\n    wandb.finish()'

In [41]:
"""if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="RobertaEmbedding-metrics",id="roberta_v1", resume="allow")
    wandb.log({
        "map_at_3_train":roberta_train,   
        "map_at_3_val":roberta_val,
        "accuracy_at_train":roberta_train_acc,
        "accuracy_at_val":roberta_val_acc,
        "f1_at_train":roberta_train_f1,
        "f1_at_val":roberta_val_f1,
        "embedding_model":"ROBERTA",
         "max_length":128})
    wandb.finish()"""

'if wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="RobertaEmbedding-metrics",id="roberta_v1", resume="allow")\n    wandb.log({\n        "map_at_3_train":roberta_train,   \n        "map_at_3_val":roberta_val,\n        "accuracy_at_train":roberta_train_acc,\n        "accuracy_at_val":roberta_val_acc,\n        "f1_at_train":roberta_train_f1,\n        "f1_at_val":roberta_val_f1,\n        "embedding_model":"ROBERTA",\n         "max_length":128})\n    wandb.finish()'

In [42]:
"""if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="SentenceTransformer-metrics",id="st_v1", resume="allow")
    wandb.log({
        "map_at_3_train":st_train,   
        "map_at_3_val":st_val,
        "accuracy_at_train":st_train_acc,
        "accuracy_at_val":st_val_acc,
        "f1_at_train":st_train_f1,
        "f1_at_val":st_val_f1,
        "embedding_model":"Sentence Transformer",
        "max_length":128})
    wandb.finish()"""

'if wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="SentenceTransformer-metrics",id="st_v1", resume="allow")\n    wandb.log({\n        "map_at_3_train":st_train,   \n        "map_at_3_val":st_val,\n        "accuracy_at_train":st_train_acc,\n        "accuracy_at_val":st_val_acc,\n        "f1_at_train":st_train_f1,\n        "f1_at_val":st_val_f1,\n        "embedding_model":"Sentence Transformer",\n        "max_length":128})\n    wandb.finish()'

In [43]:
"""from transformers import pipeline

classifier1=pipeline("zero-shot-classification",model="microsoft/deberta-base-mnli",device=0)

def zero_shot_apply(row,classifier,labels=True):
    options=[row[c] for c in option_cols]
    result=classifier(row['prompt'],candidate_labels=options)
    ranked=result['labels'][:3]
    scores=result['scores']
    label_map={row[c]:c for c in option_cols}
    if labels:
        return " ".join([label_map[opt] for opt in ranked]),scores

z1_train_results=x_train.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()
z1_train_labels=[r[0] for r in z1_train_results]
z1_train_scores=[r[1] for r in z1_train_results]

z1_val_results=x_val.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()
z1_val_labels=[r[0] for r in z1_val_results]
z1_val_scores=[r[1] for r in z1_val_results]

z1_train_map3=map_at_3(z1_train_labels,y_train)
z1_val_map3=map_at_3(z1_val_labels,y_val)"""

'from transformers import pipeline\n\nclassifier1=pipeline("zero-shot-classification",model="microsoft/deberta-base-mnli",device=0)\n\ndef zero_shot_apply(row,classifier,labels=True):\n    options=[row[c] for c in option_cols]\n    result=classifier(row[\'prompt\'],candidate_labels=options)\n    ranked=result[\'labels\'][:3]\n    scores=result[\'scores\']\n    label_map={row[c]:c for c in option_cols}\n    if labels:\n        return " ".join([label_map[opt] for opt in ranked]),scores\n\nz1_train_results=x_train.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()\nz1_train_labels=[r[0] for r in z1_train_results]\nz1_train_scores=[r[1] for r in z1_train_results]\n\nz1_val_results=x_val.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()\nz1_val_labels=[r[0] for r in z1_val_results]\nz1_val_scores=[r[1] for r in z1_val_results]\n\nz1_train_map3=map_at_3(z1_train_labels,y_train)\nz1_val_map3=map_at_3(z1_val_labels,y_val)'

In [44]:
from datasets import Dataset

label_map={'A':0,'B':1,'C':2,'D':3,'E':4}

x_tr=x_train.copy()
x_v=x_val.copy()
x_tr['label']=y_train.map(label_map)
x_v['label']=y_val.map(label_map)

train_dataset=Dataset.from_pandas(x_tr.reset_index(drop=True))
val_dataset=Dataset.from_pandas(x_v.reset_index(drop=True))

In [45]:
from transformers import AutoModelForMultipleChoice,AutoTokenizer,TrainingArguments,Trainer
import torch

model_name="distilbert-base-uncased"
tokeniser=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForMultipleChoice.from_pretrained(model_name,ignore_mismatched_sizes=True)

def preprocessing(data):
    prompts=[[p]*5 for p in data['prompt']]
    options=[[data[o][i] for o in ['A','B','C','D','E']] for i in range(len(data['prompt']))]
    prompts=sum(prompts,[])
    options=sum(options,[])
    processed=tokeniser(prompts,options,truncation=True,max_length=256,padding='max_length')
    result={k:[v[i:i+5] for i in range(0,len(v),5)] for k,v in processed.items()}
    if "label" in data:
         result["label"]=data["label"]

    return result

processed_train=train_dataset.map(preprocessing,batched=True,remove_columns=train_dataset.column_names)
processed_val=val_dataset.map(preprocessing,batched=True,remove_columns=val_dataset.column_names)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForMultipleChoice LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [46]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    top3=np.argsort(-logits,axis=1)[:,:3]
    ap=[1.0/(list(row).index(l)+1) if l in row else 0.0 for row,l in zip(top3,labels)]
    acc=(np.argmax(logits, axis=1)==labels).mean()
    preds=np.argmax(logits,axis=1)
    f1_macro=f1_score(labels,preds,average="macro")
    return {"map_at_3":np.mean(ap),"accuracy":acc,"f1_macro":f1_macro}

In [47]:
"""del model
torch.cuda.empty_cache()"""

'del model\ntorch.cuda.empty_cache()'

In [48]:
processed_train.set_format(type="torch",columns=["input_ids","attention_mask","label"])
processed_val.set_format(type="torch",columns=["input_ids","attention_mask","label"])

train_args=TrainingArguments(output_dir="./distilbert_mcq",learning_rate=2e-5,eval_strategy="epoch",save_strategy="epoch",num_train_epochs=3,warmup_steps=500,remove_unused_columns=False,max_grad_norm=1.0,
                             fp16=False,weight_decay=0.01,logging_steps=10,metric_for_best_model="map_at_3",per_device_train_batch_size=8,per_device_eval_batch_size=8,load_best_model_at_end=True)

trainer=Trainer(model=model,args=train_args,train_dataset=processed_train,eval_dataset=processed_val,compute_metrics=compute_metrics)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map At 3,Accuracy,F1 Macro
1,3.202009,3.201561,0.533750,0.355000,0.352341
2,2.940968,2.875339,0.657917,0.490000,0.486893
3,2.079963,1.877642,0.836250,0.745000,0.742634


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=300, training_loss=2.928276481628418, metrics={'train_runtime': 278.4727, 'train_samples_per_second': 17.237, 'train_steps_per_second': 1.077, 'total_flos': 1589580435456000.0, 'train_loss': 2.928276481628418, 'epoch': 3.0})

In [49]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test_dataset=Dataset.from_pandas(test)
processed_test=test_dataset.map(preprocessing,batched=True,remove_columns=test_dataset.column_names)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [50]:
"""model1=AutoModelForMultipleChoice.from_pretrained("/kaggle/working/distilbert_mcq/checkpoint-300")
trainer=Trainer(model=model1,compute_metrics=compute_metrics)"""

'model1=AutoModelForMultipleChoice.from_pretrained("/kaggle/working/distilbert_mcq/checkpoint-300")\ntrainer=Trainer(model=model1,compute_metrics=compute_metrics)'

In [51]:
train_results=trainer.evaluate(processed_train)
train_results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.728757381439209,
 'eval_map_at_3': 0.8835416666666666,
 'eval_accuracy': 0.816875,
 'eval_f1_macro': 0.8158178406198877,
 'eval_runtime': 29.7009,
 'eval_samples_per_second': 53.87,
 'eval_steps_per_second': 3.367,
 'epoch': 3.0}

In [52]:
val_results=trainer.evaluate(processed_val)
val_results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.8776415586471558,
 'eval_map_at_3': 0.83625,
 'eval_accuracy': 0.745,
 'eval_f1_macro': 0.742633822263697,
 'eval_runtime': 7.4376,
 'eval_samples_per_second': 53.781,
 'eval_steps_per_second': 3.361,
 'epoch': 3.0}

In [53]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="distilbert-finetune",id="distilbert_finetune1", resume="allow")
    wandb.log({
        "map_at_3_train":train_results['eval_map_at_3'],   
        "map_at_3_val":val_results['eval_map_at_3'],
        "accuracy_at_train":train_results['eval_accuracy'],
        "accuracy_at_val":val_results['eval_accuracy'],
        "f1_at_train":train_results['eval_f1_macro'],
        "f1_at_val":val_results['eval_f1_macro'],
        "MODEL":"DISTILBERT-FINETUNING",
         })
    wandb.finish()

wandb: setting up run distilbert_finetune1
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260706_191040-distilbert_finetune1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run distilbert-finetune
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/distilbert_finetune1
wandb: updating run metadata
wandb: uploading history steps 3-3, summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb:             MODEL DISTILBERT-FINETUNIN...
wandb: accuracy_at_train 0.81688
wandb:   accuracy_at_val 0.745
wandb:       f1_at_train 0.81582
wandb:         f1_at_val 0.74263
wandb:    map

In [54]:
test_results=trainer.predict(processed_test)
logits=test_results.predictions
letters=np.array(["A","B","C","D","E"])
top3=np.argsort(logits, axis=1)[:, ::-1][:, :3]
top3=[" ".join(letters[row]) for row in top3]
print(top3[:10])

['A E B', 'B E D', 'B E D', 'C E A', 'C A D', 'D B A', 'E C D', 'B A C', 'D C A', 'B C E']


In [55]:
"""from sklearn.preprocessing import StandardScaler

ss_tfidf=StandardScaler()
ss_w2v=StandardScaler()
train_final1=ss_tfidf.fit_transform(z1_train_scores)
train_final2=ss_w2v.fit_transform(length_train_scores)
val_final1=ss_tfidf.transform(tfidf_val_scores)
#val_final2=ss_w2v.transform(w2v_val_scores)"""

'from sklearn.preprocessing import StandardScaler\n\nss_tfidf=StandardScaler()\nss_w2v=StandardScaler()\ntrain_final1=ss_tfidf.fit_transform(z1_train_scores)\ntrain_final2=ss_w2v.fit_transform(length_train_scores)\nval_final1=ss_tfidf.transform(tfidf_val_scores)\n#val_final2=ss_w2v.transform(w2v_val_scores)'

In [56]:
def weighted_avg(final1,final2):
    best_score=0
    best_weights=[0.5,0.5]
    for first in np.arange(0,1.01,0.05):
        second=1-first
        if second<0:
            continue
        final=first*final1+second*final2
        labels=[]
        for score in final:
            scored=np.argsort(score)[::-1][:3]
            top_3=[chr(65+i) for i in scored]
            labels.append(" ".join(top_3)) 
        score_val = map_at_3(labels, y_train)
        if score_val>best_score:
            best_score=score_val
            best_weights=[first,second]
    return best_score,best_weights

In [57]:
"""ss_st=StandardScaler()
train_final3=ss_st.fit_transform(st_train_scores)
val_final3=ss_st.transform(st_val_scores)

best_score2,best_weights2=weighted_avg(train_final1,train_final3)
print(best_score2,best_weights2)"""

'ss_st=StandardScaler()\ntrain_final3=ss_st.fit_transform(st_train_scores)\nval_final3=ss_st.transform(st_val_scores)\n\nbest_score2,best_weights2=weighted_avg(train_final1,train_final3)\nprint(best_score2,best_weights2)'

In [58]:
"""final_train1=best_weights[0]*train_final1+best_weights[1]*train_final2
final_val1=best_weights[0]*val_final1+best_weights[1]*val_final2

final_train_labels1=score_to_labels(final_train1)
final_val_labels1=score_to_labels(final_val1)

comb1_train_map3=map_at_3(final_train_labels1,y_train)
comb1_val_map3=map_at_3(final_val_labels1,y_val)

print(comb1_train_map3)
print(comb1_val_map3)"""

'final_train1=best_weights[0]*train_final1+best_weights[1]*train_final2\nfinal_val1=best_weights[0]*val_final1+best_weights[1]*val_final2\n\nfinal_train_labels1=score_to_labels(final_train1)\nfinal_val_labels1=score_to_labels(final_val1)\n\ncomb1_train_map3=map_at_3(final_train_labels1,y_train)\ncomb1_val_map3=map_at_3(final_val_labels1,y_val)\n\nprint(comb1_train_map3)\nprint(comb1_val_map3)'

In [59]:
"""final_train2=best_weights2[0]*train_final1+best_weights2[1]*train_final3
final_val2=best_weights2[0]*val_final1+best_weights2[1]*val_final3

final_train_labels2=score_to_labels(final_train2)
final_val_labels2=score_to_labels(final_val2)

comb2_train_map3=map_at_3(final_train_labels2,y_train)
comb2_val_map3=map_at_3(final_val_labels2,y_val)

print(comb2_train_map3)
print(comb2_val_map3)"""

'final_train2=best_weights2[0]*train_final1+best_weights2[1]*train_final3\nfinal_val2=best_weights2[0]*val_final1+best_weights2[1]*val_final3\n\nfinal_train_labels2=score_to_labels(final_train2)\nfinal_val_labels2=score_to_labels(final_val2)\n\ncomb2_train_map3=map_at_3(final_train_labels2,y_train)\ncomb2_val_map3=map_at_3(final_val_labels2,y_val)\n\nprint(comb2_train_map3)\nprint(comb2_val_map3)'

In [60]:
"""comb1_train_acc,comb1_val_acc,comb1_train_f1,comb1_val_f1=support_metrics(final_train1,final_val1)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="word2vec_and_tfidf-ensemble-metrics",id="tfidf+w2v_v1",resume="allow")
    wandb.log({
        "map_at_3_train":comb1_train_map3,   
        "map_at_3_val":comb1_val_map3,
        "accuracy_at_train":comb1_train_acc,
        "accuracy_at_val":comb1_val_acc,
        "f1_at_train":comb1_train_f1,
        "f1_at_val":comb1_val_f1,
        "embedding_model":"Word2Vec+TFIDF",
        "best_weights":best_weights})
    wandb.finish()"""

'comb1_train_acc,comb1_val_acc,comb1_train_f1,comb1_val_f1=support_metrics(final_train1,final_val1)\n\nif wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="word2vec_and_tfidf-ensemble-metrics",id="tfidf+w2v_v1",resume="allow")\n    wandb.log({\n        "map_at_3_train":comb1_train_map3,   \n        "map_at_3_val":comb1_val_map3,\n        "accuracy_at_train":comb1_train_acc,\n        "accuracy_at_val":comb1_val_acc,\n        "f1_at_train":comb1_train_f1,\n        "f1_at_val":comb1_val_f1,\n        "embedding_model":"Word2Vec+TFIDF",\n        "best_weights":best_weights})\n    wandb.finish()'

In [61]:
"""comb2_train_acc,comb2_val_acc,comb2_train_f1,comb2_val_f1=support_metrics(final_train2,final_val2)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="SentenceTransformer_and_tfidf-ensemble-metrics",id="tfidf+st_v1",resume="allow")
    wandb.log({
        "map_at_3_train":comb2_train_map3,   
        "map_at_3_val":comb2_val_map3,
        "accuracy_at_train":comb2_train_acc,
        "accuracy_at_val":comb2_val_acc,
        "f1_at_train":comb2_train_f1,
        "f1_at_val":comb2_val_f1,
        "embedding_model":"Word2Vec+TFIDF",
        "best_weights":best_weights})
    wandb.finish()"""

'comb2_train_acc,comb2_val_acc,comb2_train_f1,comb2_val_f1=support_metrics(final_train2,final_val2)\n\nif wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="SentenceTransformer_and_tfidf-ensemble-metrics",id="tfidf+st_v1",resume="allow")\n    wandb.log({\n        "map_at_3_train":comb2_train_map3,   \n        "map_at_3_val":comb2_val_map3,\n        "accuracy_at_train":comb2_train_acc,\n        "accuracy_at_val":comb2_val_acc,\n        "f1_at_train":comb2_train_f1,\n        "f1_at_val":comb2_val_f1,\n        "embedding_model":"Word2Vec+TFIDF",\n        "best_weights":best_weights})\n    wandb.finish()'

In [62]:
"""test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test['prompt']=test['prompt'].apply(clean_text)
test['A']=test['A'].apply(clean_text)
test['B']=test['B'].apply(clean_text)
test['C']=test['C'].apply(clean_text)
test['D']=test['D'].apply(clean_text)
test['E']=test['E'].apply(clean_text)

tfidf_test_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(test['prompt']+" "+test[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_test_scores.append(np.asarray(scores).flatten())
tfidf_test_scores=np.array(tfidf_test_scores).T"""

'test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")\n\ntest[\'prompt\']=test[\'prompt\'].apply(clean_text)\ntest[\'A\']=test[\'A\'].apply(clean_text)\ntest[\'B\']=test[\'B\'].apply(clean_text)\ntest[\'C\']=test[\'C\'].apply(clean_text)\ntest[\'D\']=test[\'D\'].apply(clean_text)\ntest[\'E\']=test[\'E\'].apply(clean_text)\n\ntfidf_test_scores=[]\nfor col in [\'A\', \'B\', \'C\', \'D\', \'E\']:\n    combined=(test[\'prompt\']+" "+test[col]).to_list()\n    scores=tfidf.transform(combined).sum(axis=1)\n    tfidf_test_scores.append(np.asarray(scores).flatten())\ntfidf_test_scores=np.array(tfidf_test_scores).T'

In [63]:
"""w2v_prompts=np.vstack(test['prompt'].apply(sentence_embedding))
test_w2v_scores=[]
for col in ['A','B','C','D','E']:
    option_matrix=np.vstack(test[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(w2v_prompts,option_matrix))
    test_w2v_scores.append(scores)

test_w2v_scores=np.array(test_w2v_scores).T"""

"w2v_prompts=np.vstack(test['prompt'].apply(sentence_embedding))\ntest_w2v_scores=[]\nfor col in ['A','B','C','D','E']:\n    option_matrix=np.vstack(test[col].apply(sentence_embedding))\n    scores=np.diag(cosine_similarity(w2v_prompts,option_matrix))\n    test_w2v_scores.append(scores)\n\ntest_w2v_scores=np.array(test_w2v_scores).T"

In [64]:
#test_st_scores,st_test_labels=get_scores(test,tokenizer3,model3)

In [65]:
"""test_final1=ss_tfidf.transform(tfidf_test_scores)
test_final2=ss_w2v.transform(test_w2v_scores)
test_final3=ss_st.transform(test_st_scores)"""

'test_final1=ss_tfidf.transform(tfidf_test_scores)\ntest_final2=ss_w2v.transform(test_w2v_scores)\ntest_final3=ss_st.transform(test_st_scores)'

In [66]:
"""final_test1=best_weights[0]*test_final1+best_weights[1]*test_final2
test_labels1=[]
for score in final_test1:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65+i) for i in scored]
    test_labels1.append(" ".join(top_3))"""

'final_test1=best_weights[0]*test_final1+best_weights[1]*test_final2\ntest_labels1=[]\nfor score in final_test1:\n    scored=np.argsort(score)[::-1][:3]\n    top_3=[chr(65+i) for i in scored]\n    test_labels1.append(" ".join(top_3))'

In [67]:
"""final_test2=best_weights2[0]*test_final1+best_weights2[1]*test_final3
test_labels2=[]
for score in final_test2:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65+i) for i in scored]
    test_labels2.append(" ".join(top_3))"""

'final_test2=best_weights2[0]*test_final1+best_weights2[1]*test_final3\ntest_labels2=[]\nfor score in final_test2:\n    scored=np.argsort(score)[::-1][:3]\n    top_3=[chr(65+i) for i in scored]\n    test_labels2.append(" ".join(top_3))'

In [68]:
submission=pd.DataFrame({'id':test['id'],'Prediction':top3})
submission.to_csv('submission_model1_w2v.csv',index=False)
print("Model submission saved!")
print(submission.head())

Model submission saved!
   id Prediction
0   1      A E B
1   2      B E D
2   3      B E D
3   4      C E A
4   5      C A D
